In [110]:
%pip install pandas numpy plotly nbformat scikit-learn ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sub

import kmeans as kms
import hierarchical_clustering as hclust
import dbscan as dbscan

from sklearn.datasets import make_blobs

from plotly.subplots import make_subplots
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider
from typing import Dict, List, Tuple, Any, Optional


In [112]:
N_SAMPLES = 300
N_FEATURES = 2
N_CLUSTERS = 4

In [113]:
X, y_true = make_blobs(
    n_samples=N_SAMPLES,
    centers=N_CLUSTERS,
    n_features=N_FEATURES,
    random_state=42,
    cluster_std=0.7
)

data = pd.DataFrame(X, columns=[f'Feature {i+1}' for i in range(N_FEATURES)])


In [114]:
fig = px.scatter(
    data,
    x='Feature 1',
    y='Feature 2',
    title='Сгенерированные данные для кластеризации',
    labels={'Feature 1': 'Признак 1', 'Feature 2': 'Признак 2'},
    color=y_true.astype(str),
    category_orders={'color': [str(i) for i in range(N_CLUSTERS)]},
    hover_name=data.index,
    size_max=10
)

fig.update_layout(
    width=800,
    height=600,
    font=dict(size=12)
)

fig.show()

# Функции визаулизации

In [115]:
def visualize_clusters(data : pd.DataFrame, clusters : np.ndarray, title : str) -> None:
    data_with_clusters = data.copy()
    data_with_clusters['Cluster'] = clusters.astype(str)

    fig = px.scatter(
        data_with_clusters,
        x='Feature 1',
        y='Feature 2',
        color='Cluster',
        title=title,
        labels={'Feature 1': 'Признак 1', 'Feature 2': 'Признак 2'},
        hover_name=data_with_clusters.index,
        size_max=10
    )

    fig.update_layout(
        width=800,
        height=600,
        font=dict(size=12)
    )

    fig.show()

# Метод K-means

In [116]:
elbow_trainer = kms.ElbowTrainerImpl(
    X=X,
    ks=range(1, 11),
    km_stopper=kms.KMeansIterationStoperItersCount(100),
)

elbow_results = elbow_trainer.train()

fig_elbow = go.Figure()

fig_elbow.add_trace(
    go.Scatter(
        x=[res['k'] for res in elbow_results],
        y=[res['inertia'] for res in elbow_results],
        mode='lines+markers',
        name='Инерция',
        marker=dict(size=8),
        line=dict(width=2)
    )
)

fig_elbow.update_layout(
    title='Метод локтя для выбора оптимального количества кластеров',
    xaxis_title='Количество кластеров (k)',
    yaxis_title='Инертия',
    width=800,
    height=600,
    hovermode='x unified',
    font=dict(size=12)
)

fig_elbow.show()


In [117]:
OPTIMAL_CLUSTERS = 4

In [118]:

kmeans_final = kms.KMeansImpl(
    X=X,
    clusters_count=OPTIMAL_CLUSTERS,  
    stopper=kms.KMeansIterationStoperItersCount(100),
)
kmeans_final.fit()

clusters = np.array([kmeans_final.predict(x) for x in X])

centers = kmeans_final.centers

In [119]:
visualize_clusters(data, clusters, f'Результаты K-means кластеризации (k={OPTIMAL_CLUSTERS})')

# Распредление центров

In [120]:

center_distributors = {
    'Random': kms.KMeansCenterDestributerRandom,
    'Even': kms.KMeansCenterDestributerEven,
    # 'Hyperplane': kms.KMeansCenterDestributerEventHyperplane
}

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=X[:, 0],
        y=X[:, 1],
        mode='markers',
        name='Данные',
        marker=dict(color='lightgray', size=5),
    )
)

colors = ['red', 'green', 'blue']
symbols = ['x', 'diamond', 'circle']

for idx, (name, distributor) in enumerate(center_distributors.items()):
    centers = distributor(X, OPTIMAL_CLUSTERS).destribute()
    
    fig.add_trace(
        go.Scatter(
            x=centers[:, 0],
            y=centers[:, 1],
            mode='markers',
            name=f'Центры ({name})',
            marker=dict(
                color=colors[idx],
                size=15,
                symbol=symbols[idx],
                line=dict(width=2, color='black')
            ),
        )
    )

fig.update_xaxes(title_text="Признак 1")
fig.update_yaxes(title_text="Признак 2")

fig.update_layout(
    height=600,
    width=800,
    title_text="Сравнение методов инициализации центров K-means на одной диаграмме",
    font=dict(size=11)
)

fig.show()


In [121]:
silhouette_metric = kms.SilhouetteMetricImpl()
calinski_metric = kms.CalinskiHarabaszMetricImpl()
davies_bouldin_metric = kms.DaviesBouldinMetricImpl()

comparison_results = {}

for (name, distributor) in center_distributors.items():
    kmeans_model = kms.KMeansImpl(
        X=X,
        clusters_count=OPTIMAL_CLUSTERS,
        stopper=kms.KMeansIterationStoperItersCount(100),
        center_destributer=distributor
    )
    kmeans_model.fit()
    
    clusters = np.array([kmeans_model.predict(x) for x in X])
    centers = kmeans_model.centers
    
    comparison_results[name] = {
        'clusters': clusters,
        'centers': centers,
        'silhouette': silhouette_metric.compute(X, centers, clusters),
        'calinski': calinski_metric.compute(X, centers, clusters),
        'davies_bouldin': davies_bouldin_metric.compute(X, centers, clusters)
    }


metrics_comparison_df = pd.DataFrame({
    'Метод инициализации': list(comparison_results.keys()),
    'Silhouette': [r['silhouette'] for r in comparison_results.values()],
    'Calinski-Harabasz': [r['calinski'] for r in comparison_results.values()],
    'Davies-Bouldin': [r['davies_bouldin'] for r in comparison_results.values()]
})

print("\n=== Сравнение метрик для разных методов инициализации ===\n")
print(metrics_comparison_df.to_string(index=False))


=== Сравнение метрик для разных методов инициализации ===

Метод инициализации  Silhouette  Calinski-Harabasz  Davies-Bouldin
             Random    0.638502         972.569527        0.750612
               Even    0.854624        6909.098431        0.202885


# Иерархическая кластеризация

## Агломеративный метод

In [122]:
agglomerative : hclust.HierarchicalClustering = hclust.AgglomerativeClustering(
    X=X,
    linkage_method=hclust.AverageLinkage(),
    distance_metric=hclust.EuclideanDistance()
)

agglomerative.fit()

clusters = agglomerative.predict(OPTIMAL_CLUSTERS)


In [123]:
visualize_clusters(
    data,
    clusters,
    f'Результаты Агломеративной кластеризации (k={OPTIMAL_CLUSTERS})'
)

In [ ]:

def visualize_dendrogram_with_slider(
    clustering_model: hclust.HierarchicalClustering, 
    X: np.ndarray, 
    data: pd.DataFrame
) -> None:
    """
    Визуализирует дендрограмму с интерактивным слайдером для выбора количества кластеров
    
    Args:
        clustering_model: модель кластеризации (агломеративная или дивизивная)
        X: данные размера (n_samples, n_features)
        data: DataFrame с данными для визуализации
    """
    
    def get_dendrogram_coords(
        node: hclust.DendrogramNode, 
        x: float = 0
    ) -> Tuple[Dict[str, Any], float, Dict[int, float]]:
        """
        Вычисляет координаты для визуализации дендрограммы
        
        Args:
            node: узел дендрограммы
            x: текущая x-координата
        
        Returns:
            coords: словарь с координатами линий
            next_x: следующая доступная x-координата
            leaf_positions: позиции листьев
        """
        if node.left_child is None and node.right_child is None:
            return {'x': x, 'y': 0}, x + 1, {node.index: x}
        
        left_coords, x_after_left, left_positions = get_dendrogram_coords(node.left_child, x)
        right_coords, x_after_right, right_positions = get_dendrogram_coords(node.right_child, x_after_left)
        
        current_x = (left_coords['x'] + right_coords['x']) / 2
        current_y = node.distance
        
        
        leaf_positions = {**left_positions, **right_positions}
        
        return {
            'x': current_x,
            'y': current_y,
            'left': left_coords,
            'right': right_coords
        }, x_after_right, leaf_positions
    
    def draw_dendrogram_lines(
        node: hclust.DendrogramNode, 
        coords: Dict[str, Any], 
        x_lines: List[Optional[float]], 
        y_lines: List[Optional[float]]
    ) -> None:
        """
        Рекурсивно строит линии дендрограммы
        
        Args:
            node: узел дендрограммы
            coords: координаты узла
            x_lines: список x-координат линий
            y_lines: список y-координат линий
        """
        if 'left' not in coords:
            return
        

        x_lines.extend([coords['x'], coords['left']['x'], coords['left']['x'], None])
        y_lines.extend([coords['y'], coords['y'], coords['left']['y'], None])
        

        x_lines.extend([coords['x'], coords['right']['x'], coords['right']['x'], None])
        y_lines.extend([coords['y'], coords['y'], coords['right']['y'], None])
        
        
        draw_dendrogram_lines(node.left_child, coords['left'], x_lines, y_lines)
        draw_dendrogram_lines(node.right_child, coords['right'], x_lines, y_lines)
    
    dendrogram = clustering_model.get_dendrogram()
    coords, _, leaf_positions = get_dendrogram_coords(dendrogram)
    
    def update_plot(n_clusters: int) -> None:
        """
        Обновляет график при изменении слайдера
        
        Args:
            n_clusters: количество кластеров
        """
        clusters = clustering_model.predict(n_clusters)
        
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(
                f'Дендрограмма',
                f'Разбиение на {n_clusters} кластеров'
            ),
            horizontal_spacing=0.15
        )
        
        x_lines: List[Optional[float]] = []
        y_lines: List[Optional[float]] = []
        draw_dendrogram_lines(dendrogram, coords, x_lines, y_lines)
        
        fig.add_trace(
            go.Scatter(
                x=x_lines,
                y=y_lines,
                mode='lines',
                line=dict(color='blue', width=2),
                showlegend=False,
                hoverinfo='skip'
            ),
            row=1, col=1
        )
        
        if n_clusters > 1 and n_clusters < len(X):
        
            threshold: float = 0
       
            if isinstance(clustering_model, hclust.DivisiveClustering):
                split_distances = sorted([s['distance'] for s in clustering_model.split_history], reverse=True)
                if n_clusters - 1 < len(split_distances):
                    if n_clusters < len(split_distances):
                        threshold = (split_distances[n_clusters - 1] + split_distances[n_clusters]) / 2.0
                    else:
                        threshold = split_distances[n_clusters - 1] - 0.001
                else:
                    threshold = 0

            if isinstance(clustering_model, hclust.AgglomerativeClustering):
                n_samples = len(X)
                merge_distances = sorted([node.distance for node in clustering_model.nodes[n_samples:] 
                                        if node.left_child is not None and node.right_child is not None])
                threshold_idx = len(merge_distances) - n_clusters
                if threshold_idx >= 0 and threshold_idx < len(merge_distances):
                    if threshold_idx + 1 < len(merge_distances):
                        threshold = (merge_distances[threshold_idx] + merge_distances[threshold_idx + 1]) / 2.0
                    else:
                        threshold = merge_distances[threshold_idx] + 0.001
                else:
                    threshold = 0
            
            x_values = [x for x in x_lines if x is not None]
            fig.add_trace(
                go.Scatter(
                    x=[min(x_values), max(x_values)],
                    y=[threshold, threshold],
                    mode='lines',
                    line=dict(color='red', width=2, dash='dash'),
                    name='Уровень разреза',
                    showlegend=True
                ),
                row=1, col=1
            )
        
        data_with_clusters = data.copy()
        data_with_clusters['Cluster'] = clusters.astype(str)
        
        for cluster_id in sorted(data_with_clusters['Cluster'].unique()):
            cluster_data = data_with_clusters[data_with_clusters['Cluster'] == cluster_id]
            fig.add_trace(
                go.Scatter(
                    x=cluster_data['Feature 1'],
                    y=cluster_data['Feature 2'],
                    mode='markers',
                    name=f'Кластер {cluster_id}',
                    marker=dict(size=8),
                    showlegend=True
                ),
                row=1, col=2
            )
        
        fig.update_xaxes(title_text="Позиция", row=1, col=1)
        fig.update_yaxes(title_text="Расстояние", row=1, col=1)
        fig.update_xaxes(title_text="Признак 1", row=1, col=2)
        fig.update_yaxes(title_text="Признак 2", row=1, col=2)
        
        fig.update_layout(
            height=500,
            width=1400,
            font=dict(size=11),
        )
        
        fig.show()
    
    interact(
        update_plot,
        n_clusters=IntSlider(
            min=1,
            max=len(X),
            step=1,
            value=OPTIMAL_CLUSTERS,
            description='Кластеры:',
            continuous_update=False
        )
    )


In [125]:
visualize_dendrogram_with_slider(agglomerative, X, data)

interactive(children=(IntSlider(value=4, continuous_update=False, description='Кластеры:', max=300, min=1), Ou…

## Дивизивный метод

In [126]:
divisive = hclust.DivisiveClustering(
    X=X,
    linkage_method=hclust.AverageLinkage(),
    distance_metric=hclust.EuclideanDistance()
)

divisive.fit()

clusters = divisive.predict(OPTIMAL_CLUSTERS)

In [127]:
visualize_clusters(
    data,
    clusters,
    f'Результаты Дивизионной кластеризации (k={OPTIMAL_CLUSTERS})'
)

In [128]:
visualize_dendrogram_with_slider(divisive, X, data)

interactive(children=(IntSlider(value=4, continuous_update=False, description='Кластеры:', max=300, min=1), Ou…

# DBSCAN

In [138]:
dbscan_model = dbscan.DBSCAN(
    X=X,
    eps=2.5,
    min_samples=5,  
    distance_metric=dbscan.EuclideanDistance()
)

dbscan_model.fit()

clusters = dbscan_model.get_labels()

In [130]:
print(f"Количество кластеров: {dbscan_model.get_n_clusters()}")
print(f"Количество шумовых точек: {np.sum(clusters == -1)}")
print(f"Уникальные метки: {np.unique(clusters)}")

Количество кластеров: 5
Количество шумовых точек: 0
Уникальные метки: [0 1 2 3 4]


In [139]:
visualize_clusters(
    data,
    clusters,
    f'Результаты DBSCAN (eps={dbscan_model.eps}, min_samples={dbscan_model.min_samples})'
)

# Сравнение метрик качества кластеризации

# Метрики

In [132]:
def compute_clustering_centers(X: np.ndarray, clusters: np.ndarray) -> np.ndarray:
    """
    Вычисляет центроиды для каждого кластера
    
    Args:
        X: данные размера (n_samples, n_features)
        clusters: метки кластеров для каждой точки
    
    Returns:
        centers: центроиды кластеров размера (n_clusters, n_features)
    """
    unique_clusters = np.unique(clusters)
    # Фильтруем шумовые точки (метка -1 для DBSCAN)
    unique_clusters = unique_clusters[unique_clusters >= 0]
    
    centers = np.zeros((len(unique_clusters), X.shape[1]))
    for i, cluster_id in enumerate(unique_clusters):
        cluster_points = X[clusters == cluster_id]
        centers[i] = np.mean(cluster_points, axis=0)
    
    return centers

In [133]:
silhouette_metric = kms.SilhouetteMetricImpl()
calinski_metric = kms.CalinskiHarabaszMetricImpl()
davies_bouldin_metric = kms.DaviesBouldinMetricImpl()


metrics_results = {
    'Метод': [],
    'Silhouette': [],
    'Calinski-Harabasz': [],
    'Davies-Bouldin': []
}


kmeans_clusters = np.array([kmeans_final.predict(x) for x in X])
kmeans_centers = kmeans_final.centers

metrics_results['Метод'].append('K-means')
metrics_results['Silhouette'].append(silhouette_metric.compute(X, kmeans_centers, kmeans_clusters))
metrics_results['Calinski-Harabasz'].append(calinski_metric.compute(X, kmeans_centers, kmeans_clusters))
metrics_results['Davies-Bouldin'].append(davies_bouldin_metric.compute(X, kmeans_centers, kmeans_clusters))


agg_clusters = agglomerative.predict(OPTIMAL_CLUSTERS)
agg_centers = compute_clustering_centers(X, agg_clusters)

metrics_results['Метод'].append('Агломеративная')
metrics_results['Silhouette'].append(silhouette_metric.compute(X, agg_centers, agg_clusters))
metrics_results['Calinski-Harabasz'].append(calinski_metric.compute(X, agg_centers, agg_clusters))
metrics_results['Davies-Bouldin'].append(davies_bouldin_metric.compute(X, agg_centers, agg_clusters))


div_clusters = divisive.predict(OPTIMAL_CLUSTERS)
div_centers = compute_clustering_centers(X, div_clusters)

metrics_results['Метод'].append('Дивизивная')
metrics_results['Silhouette'].append(silhouette_metric.compute(X, div_centers, div_clusters))
metrics_results['Calinski-Harabasz'].append(calinski_metric.compute(X, div_centers, div_clusters))
metrics_results['Davies-Bouldin'].append(davies_bouldin_metric.compute(X, div_centers, div_clusters))

dbscan_clusters = dbscan_model.get_labels()
if dbscan_model.get_n_clusters() > 0:
    dbscan_centers = compute_clustering_centers(X, dbscan_clusters)

    non_noise_mask = dbscan_clusters >= 0
    X_non_noise = X[non_noise_mask]
    clusters_non_noise = dbscan_clusters[non_noise_mask]
    
    metrics_results['Метод'].append('DBSCAN')
    metrics_results['Silhouette'].append(silhouette_metric.compute(X_non_noise, dbscan_centers, clusters_non_noise))
    metrics_results['Calinski-Harabasz'].append(calinski_metric.compute(X_non_noise, dbscan_centers, clusters_non_noise))
    metrics_results['Davies-Bouldin'].append(davies_bouldin_metric.compute(X_non_noise, dbscan_centers, clusters_non_noise))

metrics_df = pd.DataFrame(metrics_results)
print("\n=== Метрики качества кластеризации ===\n")
print(metrics_df.to_string(index=False))



=== Метрики качества кластеризации ===

         Метод  Silhouette  Calinski-Harabasz  Davies-Bouldin
       K-means    0.854624        6909.098431        0.202885
Агломеративная    0.854624        6909.098431        0.202885
    Дивизивная    0.854624        6909.098431        0.202885
        DBSCAN    0.773640        5303.955826        0.255457

📊 Интерпретация метрик:
  • Silhouette: [-1, 1] - чем выше, тем лучше (оптимально > 0.5)
  • Calinski-Harabasz: [0, ∞) - чем выше, тем лучше
  • Davies-Bouldin: [0, ∞) - чем ниже, тем лучше


In [134]:

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Silhouette Score', 'Calinski-Harabasz Index', 'Davies-Bouldin Index'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
)


fig.add_trace(
    go.Bar(
        x=metrics_df['Метод'],
        y=metrics_df['Silhouette'],
        name='Silhouette',
        marker=dict(color='royalblue'),
        showlegend=False
    ),
    row=1, col=1
)


fig.add_trace(
    go.Bar(
        x=metrics_df['Метод'],
        y=metrics_df['Calinski-Harabasz'],
        name='Calinski-Harabasz',
        marker=dict(color='forestgreen'),
        showlegend=False
    ),
    row=1, col=2
)


fig.add_trace(
    go.Bar(
        x=metrics_df['Метод'],
        y=metrics_df['Davies-Bouldin'],
        name='Davies-Bouldin',
        marker=dict(color='crimson'),
        showlegend=False
    ),
    row=1, col=3
)


fig.update_yaxes(title_text="Значение", row=1, col=1)
fig.update_yaxes(title_text="Значение", row=1, col=2)
fig.update_yaxes(title_text="Значение", row=1, col=3)

fig.update_xaxes(tickangle=-45)

fig.update_layout(
    height=400,
    width=1400,
    title_text="Сравнение метрик качества кластеризации",
    font=dict(size=11)
)

fig.show()


## **1. Silhouette Score (Коэффициент силуэта)**

### Как считается:
Для каждой точки $i$:
- $a(i)$ = среднее расстояние от точки $i$ до всех других точек в **том же** кластере
- $b(i)$ = минимальное среднее расстояние от точки $i$ до точек в **ближайшем другом** кластере

Силуэт для точки: 
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

Итоговая метрика = среднее по всем точкам

### Что означает:
- **Диапазон**: [-1, 1]
- **+1**: точка идеально попала в свой кластер (далеко от других кластеров)
- **0**: точка находится на границе между кластерами
- **-1**: точка скорее всего попала не в свой кластер

### Смысл:
Показывает, насколько **компактны** кластеры внутри и насколько **разделены** между собой. Хороший результат > 0.5

---

## **2. Calinski-Harabasz Index (Индекс Калински-Харабаша)**

### Как считается:
$$CH = \frac{SS_B / (k-1)}{SS_W / (n-k)}$$

Где:
- $SS_B$ = between-cluster dispersion (межкластерная дисперсия) = сумма квадратов расстояний от центров кластеров до общего центра
- $SS_W$ = within-cluster dispersion (внутрикластерная дисперсия) = сумма квадратов расстояний точек до своих центров кластеров
- $k$ = количество кластеров
- $n$ = количество точек

### Что означает:
- **Диапазон**: [0, ∞)
- **Чем выше, тем лучше**
- По сути это F-статистика из дисперсионного анализа

### Смысл:
Отношение разброса **между** кластерами к разбросу **внутри** кластеров. Высокое значение означает, что кластеры **плотные внутри** и **далеко друг от друга**.

---

## **3. Davies-Bouldin Index (Индекс Дэвиса-Боулдина)**

### Как считается:
Для каждого кластера $i$ вычисляем:
$$R_{ij} = \frac{s_i + s_j}{d_{ij}}$$

Где:
- $s_i$ = средний радиус кластера $i$ (среднее расстояние точек до центра)
- $s_j$ = средний радиус кластера $j$
- $d_{ij}$ = расстояние между центрами кластеров $i$ и $j$

Итоговая метрика:
$$DB = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} R_{ij}$$

### Что означает:
- **Диапазон**: [0, ∞)
- **Чем ниже, тем лучше**
- Для каждого кластера ищем "наихудший случай" (самый похожий другой кластер)

### Смысл:
Измеряет среднюю "схожесть" каждого кластера с его самым похожим соседом. Низкое значение = кластеры **компактные** и **хорошо разделены**. В отличие от Calinski-Harabasz, учитывает только пары наиболее схожих кластеров.

---

## **Сравнение метрик:**

| Метрика | Лучше | Учитывает | Особенность |
|---------|-------|-----------|-------------|
| **Silhouette** | Выше (→ 1) | Все точки индивидуально | Самая интуитивная |
| **Calinski-Harabasz** | Выше (→ ∞) | Общую дисперсию | Быстрая, но чувствительна к форме |
| **Davies-Bouldin** | Ниже (→ 0) | Худшие пары кластеров | Строже оценивает разделение |
